# Layout & Buttons: feature tests

Manual tests for the `feat/layout_btns` features:

1. [Collapsible sections](#1.-Collapsible-sections): toggles, titles, starting state
2. [`hide_info_card`](#2.-hide_info_card)
3. [Frequency range](#3.-Frequency-range): `freq_min` / `freq_max`, kept across clips
4. [Buttons](#4.-Buttons): built-in icons, colours, sizes, `initial_*`, toggles, submit, images, dynamic forms
   - [4.h Layout](#4.h-Layout:-row-/-column-/-grid): `row` / `column` / `grid`
   - [4.i Button groups](#4.i-Button-groups:-a-realistic-review-form): `column` / `unique` / `delimiter`
   - [4.j Mixed layout](#4.j-Layout-with-mixed-element-types): containers of mixed element types
5. [Validation](#5.-Validation)
6. [Output](#6.-Output)
7. [Keyboard](#7.-Keyboard): j / k, Shift+Tab / Tab, form navigation, hot keys

Everything shared lives in the base config ([`config/layout-and-buttons.yaml`](annotator_config/config/layout-and-buttons.yaml))
and form ([`forms/layout-and-buttons.yaml`](annotator_config/forms/layout-and-buttons.yaml)).
Each test passes **only the settings it is testing** as a `test` dict, so what is being tested is
right there in the cell. Each section lists what to check.

In [1]:
from jupyter_bioacoustic import BioacousticAnnotator

CONFIG = 'annotator_config/config/layout-and-buttons.yaml'


def jba(**test):
    """Open the annotator on the base config plus ``test``; print any validation problems."""
    ba = BioacousticAnnotator(config=CONFIG, **test)
    result = ba.validate()
    for msg in result['errors']:
        print('❌', msg)
    for msg in result['warnings']:
        print('⚠', msg)
    ba.open()
    return ba

---
## 1. Collapsible sections

### 1.a Defaults

`test = {}`: the base config only. The base form is a quick review laid out in two `row`s:
✓ / ✗ / ? radio buttons, then a 3-state call-type toggle and a one-click ⚑ flag + submit button.

Check:
- [ ] the **info card is at the top of the playlist** (above the filter bar) and starts with **▾** (no text)
- [ ] the **play row is at the top of the spectrogram** (above Type / Res) and starts with **▾** (no text)
- [ ] the form's **title row** (QUICK REVIEW … 0 / N) starts with **▾**
- [ ] info card ▾ → hides the filter bar + clip table; the info card stays (Prev / Next still work)
- [ ] play row ▾ → hides the controls, view bar and canvas; the play row stays (▶ still plays)
- [ ] form ▾ → hides everything but that title row (progress still updates)
- [ ] forms with no title / progress row (the section 4 tests) put ▾ in the Skip / Submit row instead, and keep that row
- [ ] each ▸ expands again; the spectrogram redraws at the right size after expanding
- [ ] Tab to a ▾ and press Enter: it toggles (same as a click) and does **not** submit

In [2]:
test = {}
ba_defaults = jba(**test)

<IPython.core.display.Javascript object>

### 1.b Toggle titles

`playlist_title` / `spectrogram_title` default to no text. `True` gives the standard name; a string sets it.

Check:
- [ ] info card starts with **▾ Playlist**
- [ ] play row starts with **▾ Spectrogram (kHz)**

In [3]:
test = {
    'playlist_title': False,
    # 'spectrogram_title': 'Spectrogram (kHz)',
}
jba(**test);

<IPython.core.display.Javascript object>

### 1.c Starting collapsed

Check:
- [ ] playlist, spectrogram and form all start collapsed (▸)
- [ ] expanding the spectrogram draws it at full size (it loaded while hidden)
- [ ] expanding the form shows all of it

In [4]:
test = {
    'playlist_open': False,
    'spectrogram_open': False,
    'form_open': True,
}
jba(**test);

<IPython.core.display.Javascript object>

### 1.d Collapsed form in the reviewed view

With `duplicate_entries: false`, a submitted clip shows its saved answers (the "reviewed view")
instead of the form. Uses its own output file.

Check:
- [ ] submit a clip, then **◀ Prev** back to it: the reviewed view shows, with ▾ at the start of its REVIEWED title
- [ ] collapse the form: only the REVIEWED row stays
- [ ] **Next** to an unreviewed clip: the form is rebuilt, still collapsed, ▸ in the QUICK REVIEW row
- [ ] expand: the full form is back

In [5]:
test = {
    'duplicate_entries': False,
    'output_path': 'outputs/layout-and-buttons.reviewed.csv',
}
jba(**test);

<IPython.core.display.Javascript object>

---
## 2. `hide_info_card`

With the info card hidden, the playlist toggle gets a slim row of its own in the same place.

Check:
- [ ] no info card (no time range, no Prev / Next in that row)
- [ ] a slim row with **▾ Playlist** above the filter bar; it collapses the playlist
- [ ] second cell (no title): the slim row has just ▾
- [ ] the inline widget isn't taller than it needs to be

In [6]:
test = {
    'hide_info_card': True,
    'playlist_title': True,
}
jba(**test);

<IPython.core.display.Javascript object>

In [7]:
test = {'hide_info_card': True}
jba(**test);

<IPython.core.display.Javascript object>

---
## 3. Frequency range

`freq_min` / `freq_max` (kHz) set the frequency window the spectrogram opens with, and
the one **Reset** (or the `0` key) goes back to. Whatever frequency zoom you make is kept
when you move to another clip or change Type / Res; the time zoom resets per clip.

### 3.a `freq_min: 1`, `freq_max: 8`

Check:
- [ ] the first clip opens showing 1–8 kHz (Freq min / max boxes read 1.0 / 8.0)
- [ ] **Next ▶**: still 1–8 kHz
- [ ] type 2 / 5 into the Freq boxes (Enter) → **Next ▶** → still 2–5 kHz
- [ ] zoom box (⬚) on a band → **Next ▶** → same band; the time axis is back to the whole clip
- [ ] switch Type Linear → Mel and **Update**: the same Hz band is shown on the mel scale
- [ ] **Reset**: back to 1–8 kHz (not the full range)

In [8]:
test = {
    'freq_min': 1,
    'freq_max': 8,
}
jba(**test);

<IPython.core.display.Javascript object>

### 3.b Only `freq_max`; and no setting at all

Check:
- [ ] first cell opens at 0–6 kHz
- [ ] second cell (no setting) opens at the full range; zoom to a band, **Next ▶**: the band is kept; **Reset**: full range

In [9]:
test = {'freq_max': 6}
jba(**test);

<IPython.core.display.Javascript object>

In [10]:
test = {}
jba(**test);

<IPython.core.display.Javascript object>

### 3.c Bad values

Each of these should raise a `ValueError` (printed below, no widget).

In [11]:
for bad in ({'freq_min': 'low'}, {'freq_max': -1}, {'freq_min': 9, 'freq_max': 2}):
    try:
        BioacousticAnnotator(config=CONFIG, **bad)
        print('✗ no error for', bad)
    except ValueError as e:
        print('✓', bad, '->', e)

✓ {'freq_min': 'low'} -> freq_min must be a number (kHz), got 'low'
✓ {'freq_max': -1} -> freq_max must be a non-negative number (kHz), got -1
✓ {'freq_min': 9, 'freq_max': 2} -> freq_min (9.0) must be less than freq_max (2.0)


---
## 4. Buttons

Each test replaces the base form with its own `form_config`, so the cell shows the whole form under test.

### 4.a Every built-in icon

Ten radio-style buttons on one column (`icon`).

Check:
- [ ] ✓ green, ✗ red, ? yellow, ∅ grey, ⚑ orange, ★ yellow-on-grey, ☑ / ☐ / + / − grey
- [ ] hovering shows the `jba:` name (the `label` tooltip)
- [ ] click one: white ring, the other nine fade; click another: the ring moves
- [ ] click the pressed one again: un-pressed, nothing faded (column back to empty)
- [ ] Tab to a button and press Enter: same as clicking it (selects it; does **not** submit)
- [ ] Submit, then check `icon` in [Output](#6.-Output)

In [12]:
BUILTINS = ['check', 'x', 'unsure', 'na', 'flag', 'star', 'checked', 'unchecked', 'plus', 'minus']

test = {
    'form_config': {
        'form': [
            {'button': {'column': 'icon', 'value': name, 'icon': f'jba:{name}', 'label': f'jba:{name}'}}
            for name in BUILTINS
        ],
    },
}
jba(**test);

<IPython.core.display.Javascript object>

### 4.b Colours and sizes

Check:
- [ ] 1: built-in ✓ with a dark-blue fill (`background` overrides the green)
- [ ] 2: built-in ✗ with a yellow icon (`color` overrides the dark icon)
- [ ] 3: plain text "A" in pink on teal
- [ ] 1–3: the default 30px square; 4: `size: 18` → smaller; 5: `size: 50` → bigger; 6: `width: 90, height: 25` → wide
- [ ] the text size follows the button size

In [13]:
test = {
    'form_config': {
        'form': [
            {'button': {'column': 'pick', 'value': 1, 'icon': 'jba:check', 'background': '#1e3a8a'}},
            {'button': {'column': 'pick', 'value': 2, 'icon': 'jba:x', 'color': '#f9e2af'}},
            {'button': {'column': 'pick', 'value': 3, 'icon': 'A', 'color': '#f5c2e7', 'background': '#14b8a6'}},
            {'button': {'column': 'pick', 'value': 4, 'icon': 'S', 'size': 18}},
            {'button': {'column': 'pick', 'value': 5, 'icon': 'L', 'size': 50}},
            {'button': {'column': 'pick', 'value': 6, 'icon': 'wide', 'width': 90, 'height': 25}},
        ],
    },
}
jba(**test);

<IPython.core.display.Javascript object>

### 4.c `initial_value` and the pressed look

- `present`: two buttons, `initial_value: 'unknown'`: the column starts as "unknown", not empty
- `keep`: `icon: jba:unchecked`, `pressed_icon: jba:checked`: ☐ until pressed, then ☑
- `mark`: text "M" with `pressed_background` / `pressed_color`: only the colours change when pressed
- `plain`: `highlight: false`: no ring when pressed (its own pressed look is all you see)

Check:
- [ ] neither `present` button is pressed at first; Submit → `present` = unknown in [Output](#6.-Output)
- [ ] click **+**: white ring, **−** fades; click **+** again: back to unknown (not empty), nothing faded
- [ ] `keep` shows ☐; click → ☑ with a ring (value 1); click again → ☐ (empty)
- [ ] `mark`: pressed turns green with dark text; un-pressed back to grey
- [ ] `plain`: pressed shows ☑ but no ring
- [ ] **Next ▶**: everything back to its starting state

In [14]:
test = {
    'form_config': {
        'form': [
            {'text': 'present:'},
            {'button': {'column': 'present', 'value': 'yes', 'icon': 'jba:plus', 'initial_value': 'unknown'}},
            {'button': {'column': 'present', 'value': 'no', 'icon': 'jba:minus'}},
            {'line': True},
            {'text': 'keep:'},
            {'button': {'column': 'keep', 'value': 1, 'icon': 'jba:unchecked', 'pressed_icon': 'jba:checked'}},
            {'text': 'mark:'},
            {'button': {'column': 'mark', 'value': 1, 'icon': 'M',
                        'pressed_background': '#a6e3a1', 'pressed_color': '#1e1e2e'}},
            {'text': 'plain:'},
            {'button': {'column': 'plain', 'value': 1, 'icon': 'jba:unchecked',
                        'pressed_icon': 'jba:checked', 'highlight': False}},
        ],
    },
}
jba(**test);

<IPython.core.display.Javascript object>

### 4.d Toggles

| column | setup | starts |
|---|---|---|
| `checkbox` | 2 states, `initial_value: 0` | on ☐ (0) — the state whose value matches |
| `yes_no_unsure` | 3 states, no initial | on the first state (✓ yes) |
| `unset` | 2 states, `initial_value: null` | faded ☑, value empty; first click → ☑ (1) |
| `unset_icon` | 2 states, its own `icon: jba:na` | ∅, value empty; first click → first state |

Check:
- [ ] each starts as in the table
- [ ] each click moves to the next state and wraps around (✓ → ✗ → ? → ✓)
- [ ] no white ring / fading on toggles (the icon is the state)
- [ ] Submit without clicking: `checkbox` = 0, `yes_no_unsure` = yes, the other two empty
- [ ] **Next ▶** resets all four
- [ ] Tab to a toggle and press Enter: it moves to its next state (does **not** submit)

In [15]:
CHECKBOX_STATES = [{'value': 1, 'icon': 'jba:checked'}, {'value': 0, 'icon': 'jba:unchecked'}]

test = {
    'form_config': {
        'form': [
            {'button': {'column': 'checkbox', 'initial_value': 0, 'toggle': CHECKBOX_STATES}},
            {'button': {'column': 'yes_no_unsure', 'toggle': [
                {'value': 'yes', 'icon': 'jba:check'},
                {'value': 'no', 'icon': 'jba:x'},
                {'value': 'unsure', 'icon': 'jba:unsure'},
            ]}},
            {'button': {'column': 'unset', 'initial_value': None, 'toggle': CHECKBOX_STATES}},
            {'button': {'column': 'unset_icon', 'icon': 'jba:na', 'toggle': CHECKBOX_STATES}},
        ],
    },
}
jba(**test);

<IPython.core.display.Javascript object>

### 4.e Submit buttons and `required`

- ✓ / ✗: `submit` + `column` / `value`, `required`: one-click review (writes `verdict`, submits, moves on)
- ⚑: `submit` only (no column)
- `notes` is required: every submit button waits for it
- `verdict` is required: ⚑ and the plain **Submit** wait for it too, but ✓ / ✗ don't (clicking them fills it)

Check:
- [ ] at first ✓, ✗, ⚑ and Submit are all faded / disabled
- [ ] type notes: ✓ and ✗ enable; ⚑ and Submit stay disabled
- [ ] ✓ saves `verdict = yes` (with the notes) and moves to the next clip; ✗ saves `no`
- [ ] Tab to ✓ and press Enter: same as clicking it (saves + next clip)
- [ ] Enter in the notes box does nothing special; Enter on Skip skips (doesn't submit)

In [16]:
test = {
    'form_config': {
        'form': [
            {'textbox': {'label': 'notes', 'required': True}},
            {'line': True},
            {'button': {'column': 'verdict', 'value': 'yes', 'icon': 'jba:check', 'submit': True,
                        'required': True, 'label': 'valid + submit'}},
            {'button': {'column': 'verdict', 'value': 'no', 'icon': 'jba:x', 'submit': True,
                        'label': 'invalid + submit'}},
            {'button': {'icon': 'jba:flag', 'submit': True, 'label': 'just submit'}},
        ],
    },
}
jba(**test);

<IPython.core.display.Javascript object>

### 4.f Image buttons

Images are files read by the kernel ([`annotator_config/icons/`](annotator_config/icons)).

Check:
- [ ] `species`: bird and frog images; press one → white ring + green fill (`pressed_background`), the other fades
- [ ] `seen`: a dashed circle (`image`) that becomes the bird (`pressed_image`) while pressed
- [ ] `has_bird`: a toggle between the bird image and ☐
- [ ] a missing image (`nope.svg`) shows its label ("missing") and a ❌ in the status line

In [17]:
ICONS = 'annotator_config/icons'

test = {
    'form_config': {
        'form': [
            {'button': {'column': 'species', 'value': 'bird', 'image': f'{ICONS}/bird.svg',
                        'pressed_background': '#a6e3a1', 'label': 'bird', 'size': 60}},
            {'button': {'column': 'species', 'value': 'frog', 'image': f'{ICONS}/frog.svg',
                        'pressed_background': '#a6e3a1', 'label': 'frog', 'size': 60}},
            {'button': {'column': 'seen', 'value': 1, 'image': f'{ICONS}/blank.svg',
                        'pressed_image': f'{ICONS}/bird.svg', 'label': 'seen', 'size': 60}},
            {'line': True},
            {'button': {'column': 'has_bird', 'initial_value': 0, 'toggle': [
                {'value': 1, 'image': f'{ICONS}/bird.svg'},
                {'value': 0, 'icon': 'jba:unchecked'},
            ]}},
            {'button': {'column': 'broken', 'value': 1, 'image': f'{ICONS}/nope.svg', 'label': 'missing'}},
        ],
    },
}
jba(**test);

<IPython.core.display.Javascript object>

### 4.f+ Enter on a checkbox

Check:
- [ ] Tab to the checkbox and press Enter: it toggles (same as a click) and does **not** submit
- [ ] Space still toggles it too
- [ ] Tab to **Submit** and press Enter: submits

In [18]:
test = {
    'form_config': {
        'form': [
            {'checkbox': {'label': 'flag for expert review', 'column': 'flagged',
                          'checked_value': 'yes', 'unchecked_value': 'no'}},
        ],
    },
}
jba(**test);

<IPython.core.display.Javascript object>

### 4.g Buttons in a dynamic form

Check:
- [ ] "is valid?" = no shows the correction form with its buttons
- [ ] the buttons work there; switching back to yes hides them (their values are still reset per clip)

In [19]:
test = {
    'form_config': {
        'form': [
            {'select': {'label': 'is valid?', 'column': 'is_valid', 'required': True,
                        'items': [{'value': 'yes'}, {'value': 'no', 'form': 'correction'}]}},
        ],
        'dynamic_forms': {
            'correction': [
                {'text': 'what is it?'},
                {'button': {'column': 'correction', 'value': 'noise', 'icon': 'jba:na', 'label': 'noise'}},
                {'button': {'column': 'correction', 'value': 'other species', 'icon': 'jba:unsure', 'label': 'other species'}},
            ],
        },
    },
}
jba(**test);

<IPython.core.display.Javascript object>

### 4.h Layout: `row` / `column` / `grid`

Containers nest. `row` puts its items side by side (wrapping), `column` stacks them, `grid`
fills `columns` aligned columns. Each takes a list, or `{items, gap, align}` (`grid` also `columns`).

#### Column of rows (3 / 2 / 3), and a grid

Check:
- [ ] left: three lines of 3, 2 and 3 buttons, left-aligned
- [ ] right of it (`row` of the two), a 3-column grid of 8 buttons: rows of 3, 3, 2 with columns lined up
- [ ] all 16 buttons are one radio group on `pad`: selecting any one deselects the others

In [20]:
def pad(*values):
    """Buttons on the ``pad`` column labelled with their values."""
    return [{'button': {'column': 'pad', 'value': v, 'icon': str(v)}} for v in values]

test = {
    'form_config': {
        'form': [
            {'row': {'gap': 40, 'items': [
                {'column': [
                    {'row': pad(1, 2, 3)},
                    {'row': pad(4, 5)},
                    {'row': pad(6, 7, 8)},
                ]},
                {'grid': {'columns': 3, 'items': pad('a', 'b', 'c', 'd', 'e', 'f', 'g', 'h')}},
            ]}},
        ],
    },
}
jba(**test);

<IPython.core.display.Javascript object>

#### Mixed inputs, `align`, `gap`, and a revealed form

Check:
- [ ] row 1: checkbox, textbox and select side by side (labels stay with their inputs)
- [ ] row 2 is centered (`align: center`) with a big gap (`gap: 30`)
- [ ] a grid of label + input pairs lines up in 2 columns even though the inputs differ in width
- [ ] picking "other" in the select shows the "why" form **below row 1**, not squeezed inside it
- [ ] Submit writes every input's column (see [Output](#6.-Output))

In [21]:
test = {
    'form_config': {
        'form': [
            {'row': [
                {'checkbox': {'label': 'confident', 'checked_value': 'yes', 'unchecked_value': 'no'}},
                {'textbox': {'label': 'notes'}},
                {'select': {'label': 'kind', 'items': [{'value': 'song'}, {'value': 'other', 'form': 'why'}]}},
            ]},
            {'row': {'align': 'center', 'gap': 30, 'items': [
                {'button': {'column': 'quality', 'value': 'good', 'icon': 'jba:star'}},
                {'button': {'column': 'quality', 'value': 'bad', 'icon': 'jba:na'}},
            ]}},
            {'grid': {'columns': 2, 'items': [
                {'number': {'label': 'count', 'min': 0, 'max': 20, 'step': 1}},
                {'textbox': {'label': 'observer', 'width': 160}},
                {'checkbox': {'label': 'chorus'}},
                {'select': {'label': 'distance', 'items': ['near', 'mid', 'far']}},
            ]}},
        ],
        'dynamic_forms': {
            'why': [{'textbox': {'label': 'why other?'}}],
        },
    },
}
jba(**test);

<IPython.core.display.Javascript object>

### 4.i Button groups: a realistic review form

A container's `column` is passed on to the buttons inside that don't set their own; a container
with `unique` makes all the buttons inside it (nested containers too, unless they set `unique`)
one group. Title + progress tracker, so the form's ▾ is on the title row.

| row | setup | output |
|---|---|---|
| is valid? | `row`, `column: is_valid` (unique by default), `required` | `yes` / `no` / `unsure` |
| call types | `grid`, `column: call_types`, `unique: false`, `delimiter: ';'`, `initial_value: song` | e.g. `song;alarm` (starts with song pressed) |
| loudest other sound | `row`, `unique: true`, one column per button (`value: 1`, `initial_value: 0`) | one-hot: `other_bird` … `other_noise`, exactly one 1 (or all 0) |
| recording issues | `row`, `unique: false`, one column per button | independent `clipped` / `overlapping` = `y` |
| distance | `column` with `column: distance`, `unique: true`, holding two `row`s | one of near / mid / far / unknown across both rows |

Check:
- [ ] the playlist and the first clip load; the form's ▾ is on the REVIEW row
- [ ] **Submit** is disabled until "is valid?" is answered
- [ ] call types: several can be pressed (white ring, nothing fades); `song` starts pressed
- [ ] loudest other sound / distance: pressing one un-presses and fades the others
- [ ] recording issues: both can be on
- [ ] Submit → next clip, everything reset (song pressed again); check the new row in [Output](#6.-Output)

In [2]:
def btn(value, label=None, **extra):
    """A text button (``label`` is shown and is the tooltip; defaults to the value)."""
    label = label or str(value)
    return {'button': {'value': value, 'icon': label, 'label': label, 'width': 60, **extra}}

def one_hot(col, label):
    """1 when pressed, else 0."""
    return {'button': {'column': col, 'value': 1, 'initial_value': 0, 'icon': label,
                       'label': label, 'width': 60}}

test = {
    'form_config': {
        'title': {'value': 'REVIEW DETECTION', 'progress_tracker': True},
        'form': [
            {'row': {'column': 'is_valid', 'items': [
                {'text': 'Is this a [[common_name]]?'},
                {'button': {'value': 'yes', 'icon': 'jba:check', 'label': 'yes', 'required': True}},
                {'button': {'value': 'no', 'icon': 'jba:x', 'label': 'no'}},
                {'button': {'value': 'unsure', 'icon': 'jba:unsure', 'label': 'unsure'}},
            ]}},
            {'row': [
                {'text': 'call types:'},
                {'grid': {'columns': 3, 'column': 'call_types', 'unique': False, 'delimiter': ';',
                          'items': [btn('song', initial_value='song'), btn('call'), btn('alarm'),
                                    btn('drum'), btn('flight'), btn('juvenile')]}},
            ]},
            {'row': [
                {'text': 'loudest other sound:'},
                {'row': {'unique': True, 'items': [
                    one_hot('other_bird', 'bird'), one_hot('other_frog', 'frog'),
                    one_hot('other_insect', 'insect'), one_hot('other_noise', 'noise'),
                ]}},
            ]},
            {'row': [
                {'text': 'recording issues:'},
                {'row': {'unique': False, 'items': [
                    btn('y', 'clipped', column='clipped', pressed_background='#f38ba8'),
                    btn('y', 'overlap', column='overlapping', pressed_background='#f38ba8'),
                ]}},
            ]},
            {'row': [
                {'text': 'distance:'},
                {'column': {'column': 'distance', 'unique': True, 'gap': 4, 'items': [
                    {'row': [btn('near'), btn('mid')]},
                    {'row': [btn('far'), btn('unknown')]},
                ]}},
            ]},
        ],
    },
}
jba(**test);

<IPython.core.display.Javascript object>

### 4.j Layout with mixed element types

Containers hold any element, not just buttons: a `grid` of select / number / multiselect /
checkbox, a `row` mixing text, buttons, a toggle and a checkbox, a `column` of text + a
multiline textbox, and a right-aligned `row` whose select reveals a dynamic form that itself
uses a `row`.

Check:
- [ ] the grid's labels + inputs line up in 2 columns (filter box on species; tags on behaviours)
- [ ] row 2: text, L / M / H buttons, the "present" toggle and the "needs expert" checkbox on one line
- [ ] the notes text shows this clip's common name and confidence; the textbox sits under it
- [ ] the last row is right-aligned; "is valid?" = no shows the correction form **below** it, laid out in a row
- [ ] collapse the form (▾ on the SURVEY row): only that row stays
- [ ] Submit writes every column (species, count, behaviours, juvenile, confidence, present,
      expert, notes, is_valid, flag, corrected, correction_sure) — see [Output](#6.-Output)

In [23]:
SPECIES = {'path': 'data/species_counts.N-dse.csv', 'value': 'common_name', 'filter_box': True}

test = {
    'form_config': {
        'title': {'value': 'SURVEY CLIP', 'progress_tracker': True},
        'form': [
            {'grid': {'columns': 2, 'gap': 10, 'items': [
                {'select': {'label': 'species', 'items': SPECIES}},
                {'number': {'label': 'count', 'min': 0, 'max': 50, 'step': 1, 'value': 1}},
                {'multiselect': {'label': 'behaviours',
                                 'items': ['singing', 'calling', 'foraging', 'flying']}},
                {'checkbox': {'label': 'juvenile', 'checked_value': 'yes', 'unchecked_value': 'no'}},
            ]}},
            {'row': [
                {'text': 'confidence:'},
                {'row': {'column': 'confidence', 'gap': 4, 'items': [
                    {'button': {'value': 'low', 'icon': 'L', 'label': 'low'}},
                    {'button': {'value': 'medium', 'icon': 'M', 'label': 'medium'}},
                    {'button': {'value': 'high', 'icon': 'H', 'label': 'high'}},
                ]}},
                {'text': 'present:'},
                {'button': {'column': 'present', 'initial_value': 0, 'toggle': [
                    {'value': 1, 'icon': 'jba:checked'}, {'value': 0, 'icon': 'jba:unchecked'},
                ]}},
                {'checkbox': {'label': 'needs expert', 'column': 'expert'}},
            ]},
            {'column': {'gap': 4, 'items': [
                {'text': 'notes on this [[common_name]] (model confidence [[confidence]]):'},
                {'textbox': {'label': 'notes', 'multiline': True, 'width': 420}},
            ]}},
            {'row': {'align': 'end', 'items': [
                {'select': {'label': 'is valid?', 'column': 'is_valid',
                            'items': [{'value': 'yes'}, {'value': 'no', 'form': 'correction'}]}},
                {'button': {'column': 'flag', 'value': True, 'icon': 'jba:flag', 'label': 'flag'}},
            ]}},
        ],
        'dynamic_forms': {
            'correction': [
                {'row': [
                    {'select': {'label': 'corrected species', 'column': 'corrected', 'items': SPECIES}},
                    {'text': 'sure?'},
                    {'button': {'column': 'correction_sure', 'toggle': [
                        {'value': 'yes', 'icon': 'jba:check'}, {'value': 'no', 'icon': 'jba:unsure'},
                    ]}},
                ]},
            ],
        },
    },
}
jba(**test);

<IPython.core.display.Javascript object>

---
## 5. Validation

A form with one mistake per button. Each should be reported (no widget is opened).

In [24]:
bad_form = {
    'form': [
        {'button': {'column': 'a'}},                                       # no icon/image; column without value
        {'button': {'icon': 'x'}},                                         # does nothing
        {'button': {'icon': 'jba:nope', 'submit': True}},                  # unknown built-in
        {'button': {'icon': 'x', 'submit': True, 'color': 3}},             # color not a string
        {'button': {'column': 't', 'submit': True,                         # toggle + submit, 1 state
                    'toggle': [{'value': 1, 'icon': 'a'}]}},
        {'button': {'column': 't2', 'toggle': [{'icon': 'a'}, {'value': 0}]}},  # state missing value / icon
        {'grid': [{'textbox': {'label': 'g'}}]},                          # grid without columns
        {'row': {'align': 'middle', 'items': []}},                         # bad align
    ],
}
result = BioacousticAnnotator(config=CONFIG, form_config=bad_form).validate()
print('valid:', result['valid'])
for msg in result['errors']:
    print('❌', msg)

valid: False
❌ Button "a": set "icon" (text, a symbol or a jba: icon) or "image" (a path)
❌ Button "a": "column" needs a "value" to write
❌ Button "x": set "column" + "value", "submit: true", or both
❌ Button "jba:nope" icon: unknown built-in icon "jba:nope" (use jba:check, jba:checked, jba:flag, jba:minus, jba:na, jba:plus, jba:star, jba:unchecked, jba:unsure, jba:x)
❌ Button "x": "color" must be a CSS color string
❌ Button "t": a toggle can't "submit" (add a separate submit button)
❌ Button "t": "toggle" must be a list of 2 or more states
❌ Button "t2" toggle state 1: needs a "value"
❌ Button "t2" toggle state 2: set "icon" or "image"
❌ "grid" needs "columns" and "items" (e.g. {columns: 3, items: [...]})
❌ "row": "align" must be one of start, center, end


---
## 6. Output

The rows submitted from every widget above (except 1.d, which writes its own file). Button
columns hold the button values; toggles their state values; untouched buttons with no
`initial_value` are empty.

In [25]:
ba_defaults.output(force=True).tail(20)

,detection_id,is_valid,call_type,flagged,icon,notes,verdict,pad,tags,owl,cat,frog,day,night,size,x,y,picks,sp,v
0,5780,unsure,call,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5780,NaN,NaN,NaN,plus,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5780,NaN,NaN,NaN,NaN,asd,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1191,NaN,NaN,NaN,NaN,SDAF,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5429,NaN,NaN,NaN,NaN,AFDSA,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5780,yes,song,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,5780,yes,call,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,5780,NaN,NaN,NaN,NaN,NaN,NaN,a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,5780,NaN,NaN,NaN,NaN,NaN,NaN,NaN,a;c,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,"b,c",NaN,NaN


---
## 7. Keyboard

### 7.a Moving around: j / k, Shift+Tab, Tab

Uses the base form (section 1.a). Click anywhere in the widget first (e.g. a clip table row).

Check:
- [ ] **k** → next clip, **j** → previous clip, from the clip table, the spectrogram and the form
- [ ] j / k do **not** move the notebook's cells
- [ ] j / k type normally in a text box (no clip change)
- [ ] **Shift+Tab** cycles: playlist (clip table, blue outline) → spectrogram (canvas) → form → playlist …
- [ ] collapse the spectrogram (▾ on the play row), then Shift+Tab: the play row's ▸ gets focus instead of the canvas
- [ ] **Tab** from the playlist / spectrogram jumps into the form (first item outlined)

In [2]:
test = {}
jba(**test);

<IPython.core.display.Javascript object>

### 7.b Form navigation and hot keys

| item | keys |
|---|---|
| species select | `hot_keys` 1 / 2 / 3, `pin_hot_keys` (the list starts with a "hot keys" group) |
| is valid? row | ✓ `y`, ✗ `n`, ? `u` (`show_hot_key: true`: letters in the corners) |
| call types grid (`unique: false`) | `s` song, `c` call, `a` alarm — press again to un-press |
| flags row | `j` = "juvenile" (overrides j = previous clip **only while this row is focused**); `show_hot_key: false` on "faint" (`f`) |
| notes textbox | — |
| quick submit row | ⚑ `q`: flag + submit |

Check:
- [ ] Tab into the form: the species select is outlined; **1** picks Tree Swallow, **2** Spotted Towhee, **3** Song Sparrow
- [ ] open the species list: the "hot keys" group is first, then "all"
- [ ] **Tab** / **→** to the is-valid row (the whole row is outlined): **y** / **n** / **u** press the buttons; pressing again un-presses
- [ ] keys only work in their item: on the is-valid row, **s** does nothing; on the call-types grid, **y** does nothing
- [ ] call types: **s** then **a** → both pressed (`song,alarm`)
- [ ] flags row: **j** presses "juvenile" (no clip change); **f** presses "faint" (no badge on it); leave the row (Tab) and **j** goes to the previous clip again
- [ ] **Enter** on a row steps into it (first button focused), **← / →** move inside, **Esc** steps back out (row outlined)
- [ ] notes: Tab puts the cursor in; type freely (j / k / arrows type); **Esc** leaves (label outlined); **Tab** leaves too
- [ ] **Space** / **Enter** on a focused button press it
- [ ] **q** on the quick-submit row: saves (flag = true) and moves to the next clip; focus is back on the species select
- [ ] **← / →** past the last / first item wraps around

In [7]:
test = {
    'form_config': {
        'title': {'value': 'KEYBOARD REVIEW', 'progress_tracker': True},
        'form': [
            {'select': {'label': 'species', 'column': 'species',
                        'hot_keys': {1: 'Tree Swallow', 2: 'Spotted Towhee', 3: 'Song Sparrow'},
                        'pin_hot_keys': True,
                        'items': {'path': 'data/species_counts.N-dse.csv', 'value': 'common_name'}}},
            {'row': {'column': 'is_valid', 'items': [
                {'text': 'is valid?'},
                {'button': {'value': 'yes', 'icon': 'jba:check', 'hot_key': 'y', 'show_hot_key': True}},
                {'button': {'value': 'no', 'icon': 'jba:x', 'hot_key': 'n', 'show_hot_key': True}},
                {'button': {'value': 'unsure', 'icon': 'jba:unsure', 'hot_key': 'u', 'show_hot_key': True}},
            ]}},
            {'row': [
                {'text': 'call types:'},
                {'grid': {'columns': 3, 'column': 'call_types', 'unique': False, 'items': [
                    {'button': {'value': 'song', 'icon': 'song', 'width': 50, 'hot_key': 's'}},
                    {'button': {'value': 'call', 'icon': 'call', 'width': 50, 'hot_key': 'c'}},
                    {'button': {'value': 'alarm', 'icon': 'alarm', 'width': 50, 'hot_key': 'a'}},
                ]}},
            ]},
            {'row': {'unique': False, 'items': [
                {'text': 'flags:'},
                {'button': {'column': 'juvenile', 'value': 'y', 'icon': 'juv', 'width': 40, 'hot_key': 'j'}},
                {'button': {'column': 'faint', 'value': 'y', 'icon': 'faint', 'width': 40, 'hot_key': 'f',
                            'show_hot_key': False}},
            ]}},
            {'textbox': {'label': 'notes'}},
            {'row': [
                {'text': 'quick submit:'},
                {'button': {'column': 'flag', 'value': True, 'icon': 'jba:flag', 'hot_key': 'q',
                            'submit': True, 'label': 'flag + submit'}},
            ]},
        ],
    },
}
jba(**test);

<IPython.core.display.Javascript object>